# PeptiScout AI — Midterm Report

**Course:** CPS 5801 — Advanced AI Systems (Kean University)  
**Project:** PeptiScout — reasoning-capable peptide information assistant with RAG, tools, and planned fine-tuning.

*Add your name and submission date in this cell if required by your instructor.*

*This notebook summarizes work completed to date, key artifacts, runnable demos, and the roadmap through the final deliverable.*


## 1. Problem and goals

General-purpose LLMs often **hallucinate** clinical peptide details (dosing, mechanisms, citations). Static calculators lack reasoning. **PeptiScout** targets:

- Precise **reconstitution math** (deterministic, no LLM).
- **Mechanism-of-action** explanations with pathway-level detail where evidence exists.
- **PubMed-grounded** retrieval (RAG) for auditability.
- **Vendor / source vetting** and **lab image** analysis (planned wiring in full agent).

The full system design (modes, tools, evaluation rubric) is documented in `PEPTISCOUT_CURSOR_CONTEXT.md` at the repository root.


## 2. Architecture (high level)

| Layer | Technology |
|---|---|
| Frontend | React + Vite + Tailwind (`frontend/`) |
| Backend | FastAPI (`backend/main.py`) |
| Agent (planned) | LangGraph ReAct graph (`backend/agent/graph.py`) |
| Baseline LLM | OpenAI GPT-4o family (via API) |
| Embeddings / RAG | `text-embedding-3-small`, Pinecone |
| PubMed | NCBI E-utilities (no key; email in env) |

**Structured response (target contract):** `protocol`, `moa`, `good_bad`, `audit_trail`, and optionally `react_trace` for the full agent.


## 3. What is implemented in this repository (midterm snapshot)

### Done or in progress

- **FastAPI app** with CORS, health check `GET /api/health`, and **Tool A** exposed as `POST /api/calculate` (`backend/routers/tools.py`).
- **Tool A — calculator** — pure Python reconstitution math (`backend/tools/calculator.py`).
- **Dataset / PubMed pipeline** — `backend/scripts/generate_dataset.py` implements phased work: fetch abstracts from NCBI, build **Alpaca-style** instruction rows with a teacher model (`gpt-4o-mini`), optional Pinecone ingest; uses `backend/data/raw_abstracts.json` and **`peptide_dataset_checkpoint.json`** for resumable generation.
- **Tool modules** present for RAG, VLM bloodwork, Tavily vetting (`backend/tools/rag_retriever.py`, `vlm_analyzer.py`, `source_vetter.py`) — to be fully wired per project plan.

### Stubs / next steps

- **`backend/routers/query.py`** — router file exists; full `POST /api/query` modes (baseline / fine-tuned / full-agent) to be completed.
- **`backend/agent/graph.py`** — `build_graph()` placeholder (`NotImplementedError`); LangGraph ReAct graph is Step 8 in the master plan.
- **`backend/routers/results.py`** — stub; will serve `results.json` for the Results page after evaluation runs.
- **`backend/notebooks/`** — reserved for `peptiscout_finetuning.ipynb` and `peptiscout_evaluation.ipynb` per rubric (not yet added; see roadmap below).


## 4. Data artifacts (this checkout)

Loads JSON from `backend/data/` and prints summary statistics. Set the working directory to the **repository root** (`PeptiScout`) before running.


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
DATA = ROOT / "backend" / "data"
print("Repository root (cwd):", ROOT.resolve())
print("Data directory exists:", DATA.is_dir())

RAW = DATA / "raw_abstracts.json"
if RAW.is_file():
    with open(RAW, encoding="utf-8") as f:
        raw_doc = json.load(f)
    recs = raw_doc.get("records", [])
    print("\nraw_abstracts.json")
    print("  version:", raw_doc.get("version"))
    print("  records:", len(recs))
    if recs:
        print("  sample keys:", list(recs[0].keys()))
else:
    print("\nraw_abstracts.json not found at", RAW)

CK = DATA / "peptide_dataset_checkpoint.json"
if CK.is_file():
    with open(CK, encoding="utf-8") as f:
        ck = json.load(f)
    rows = ck.get("rows", [])
    print("\npeptide_dataset_checkpoint.json")
    print("  Alpaca rows:", len(rows))
    if rows:
        print("  fields per row:", list(rows[0].keys()))
        print("  sample _meta:", rows[0].get("_meta", {}))
else:
    print("\npeptide_dataset_checkpoint.json not found at", CK)


## 5. Runnable demo — Tool A (reconstitution calculator)

Same logic as the API; imports from `backend.tools.calculator`. Requires cwd = repo root so `backend` is importable.


In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from backend.tools.calculator import CalculateRequest, calculate_reconstitution

req = CalculateRequest(vial_mg=5, water_mL=2, dose_mcg=250, syringe_type="U100")
resp = calculate_reconstitution(req)
print(resp.model_dump())


## 6. How to run the dataset script (from project root)

```bash
# Phases: fetch (PubMed), alpaca (teacher pairs), pinecone (vectors) — see script help
python -m backend.scripts.generate_dataset --help
```

Requires `.env` with `OPENAI_API_KEY`, `NCBI_EMAIL`, and for Pinecone phases the Pinecone variables (see `.env.example`). **Do not commit secrets.**


## 7. Roadmap to final (from project build order)

Aligned with `PEPTISCOUT_CURSOR_CONTEXT.md`:

1. Complete **`POST /api/query`** — baseline modes (`baseline-zero-shot`, `baseline-few-shot`, `baseline-cot`), then fine-tuned and full-agent.
2. **LangGraph** — implement `build_graph()` and `mode=full-agent` with ReAct trace.
3. **Fine-tuning notebook** — LoRA on Llama-3-8B per spec; run on Colab; place adapter under `backend/models/peptide_lora_adapter/`.
4. **Evaluation** — `benchmark_100.json`, metrics (DS, PC, TSR, CA), ablations, export `results.json`.
5. **Frontend** — Demo page mode switcher, Results page charts, remaining pages as specified.

Items explicitly **manual** for the course (IEEE report, slides/video, full benchmark curation, etc.) stay out of automated code paths per the master doc.


## 8. API surface (reference)

| Method | Path | Status (midterm) |
|---|---|---|
| GET | `/api/health` | Implemented |
| POST | `/api/calculate` | Implemented |
| POST | `/api/query` | Planned |
| POST | `/api/analyze-bloodwork`, `/api/vetting` | Per master plan |
| GET | `/api/results` | Planned |
